### **Sieć neuronowa MTL na reprezentacji fingerprint - zestaw 2 - Eliminacja**

Wykorzystana reprezentacja: **ECFP4**

Lista endpointów:


1. Half Life (Obach)
2. Clearance Hepatocyte (AstraZeneca)
3. CYP3A4 Inhibition (Veith)
4. VDss (Lombardo)
5. AMES Mutagenicity

Wyniki dla STL:

In [1]:
!pip install rdkit
!pip install pandas numpy scikit-learn -U
!pip install pytdc --no-dependencies
!pip install fuzzywuzzy

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
data_folder = "/content/drive/MyDrive/mldd_data" #folder z zapisanymi zestawami danych aby umożliwić porównanie danych na dokładnie tych samych zestawach dla każdego modelu

#data_folder = "/content/drive/MyDrive/MLDD - ADMET/data_splits"

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from tdc.single_pred import ADME
from rdkit import Chem
from rdkit.Chem import AllChem
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, roc_auc_score, accuracy_score, f1_score

In [5]:
class Featurizer:
    def __init__(self, y_column, smiles_col='Drug', **kwargs):
        self.y_column = y_column
        self.smiles_col = smiles_col
        self.__dict__.update(kwargs)
    def __call__(self, df):
        raise NotImplementedError()

class ECFPFeaturizer(Featurizer): #dodane zabezpieczenia na Nan etc
    def __init__(self, y_column='Y', radius=2, length=1024, **kwargs):
        self.radius = radius
        self.length = length
        super().__init__(y_column, **kwargs)

    def __call__(self, df):
        fingerprints = []
        labels = []
        for i, row in df.iterrows():
            smiles = row[self.smiles_col]
            mol = Chem.MolFromSmiles(str(smiles)) if pd.notna(smiles) else None

            if mol:
                fp = AllChem.GetMorganFingerprintAsBitVect(mol, self.radius, nBits=self.length)
                fingerprints.append(np.array(fp))
            else:
                # Zamiast pomijać wiersz, wstawiamy wektor zer (zachowuje wyrównanie)
                fingerprints.append(np.zeros(self.length))

            # Pobieramy etykietę jeśli istnieje (dla dummy X wstawiamy NaN)
            labels.append(row[self.y_column] if self.y_column in df.columns else np.nan)

        return np.array(fingerprints), np.array(labels).reshape(-1, 1)

In [6]:

# DEFINICJA SIECI NEURONOWEJ

class AdmetEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim=512, dropout=0.2):
        super().__init__()
        self.main = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        self.res_layer = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, x):
        x = self.main(x)
        return torch.relu(x + self.res_layer(x))


class MTL_NN_Hybrid(nn.Module):
    def __init__(self, input_dim, reg_tasks, class_tasks, hidden_dim=512):
        """
        reg_tasks: lista nazw zadań regresyjnych (np. ['Lipophilicity', 'Solubility'])
        class_tasks: lista nazw zadań klasyfikacyjnych (np. ['BBB', 'Ames'])
        """
        super().__init__()
        self.encoder = AdmetEncoder(input_dim, hidden_dim)
        self.reg_tasks = reg_tasks
        self.class_tasks = class_tasks

        # Słowniki głowic dla każdego typu zadania
        #Pozwala zapisać wiele warstw Linear i nazwać je tak, jak nazywają się Twoje zadania.
        self.reg_heads = nn.ModuleDict({
            task: nn.Linear(hidden_dim, 1) for task in reg_tasks
        })
        self.class_heads = nn.ModuleDict({
            task: nn.Linear(hidden_dim, 1) for task in class_tasks
        })

    def forward(self, x):
        shared_features = self.encoder(x) #wspólny enkoder
        results = {}

        # Procesowanie zadań regresyjnych
        for task in self.reg_tasks:
            results[task] = self.reg_heads[task](shared_features)

        # Procesowanie zadań klasyfikacyjnych
        for task in self.class_tasks:
            # results[task] = torch.sigmoid(self.class_heads[task](shared_features))
            results[task] = self.class_heads[task](shared_features)

        #print(results)
        return results #Zwracając słownik:{'Caco2_Wang': 1.2, 'Lipophilicity_AZ': 0.5} masz absolutną pewność, który wynik dotyczy którego badania.

In [7]:

# --- STL REGRESOR ---
class STL_NN_Regressor(nn.Module):
    def __init__(self, input_dim, hidden_dim=512):
        super().__init__()
        self.encoder = AdmetEncoder(input_dim, hidden_dim)
        self.head = nn.Linear(hidden_dim, 1) # Wynik liniowy (dowolna liczba)

    def forward(self, x):
        return self.head(self.encoder(x))

# --- STL KLASYFIKATOR ---
class STL_NN_Classifier(nn.Module):
    def __init__(self, input_dim, hidden_dim=512):
        super().__init__()
        self.encoder = AdmetEncoder(input_dim, hidden_dim)
        self.head = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        # Sigmoid zamienia wynik na prawdopodobieństwo (0-1)
        return torch.sigmoid(self.head(self.encoder(x)))

  #=============================

In [8]:
def train_MTL_hybrid_wl_sum(X_train_np, X_test_np, y_train_dict, y_test_dict, reg_tasks, class_tasks):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # --- 1. Normalizacja (osobny skaler dla każdego zadania regresyjnego) ---
    scalers = {}
    y_train_tensors = {}
    y_test_tensors = {}

    for task in reg_tasks:
        scalers[task] = StandardScaler()
        # Wyciągamy dane i maskujemy NaN, aby skaler zadziałał
        train_vals = y_train_dict[task].reshape(-1, 1)
        mask = ~np.isnan(train_vals).flatten()

        y_train_scaled = np.full_like(train_vals, np.nan)
        if mask.any():
            y_train_scaled[mask] = scalers[task].fit_transform(train_vals[mask])

        y_train_tensors[task] = torch.FloatTensor(y_train_scaled).to(device)
        # Testowe przesyłamy w oryginale (skalujemy tylko przy ewaluacji dla wygody)
        y_test_tensors[task] = torch.FloatTensor(y_test_dict[task].reshape(-1, 1)).to(device)

    for task in class_tasks:
        # Klasyfikacja nie wymaga skalowania
        y_train_tensors[task] = torch.FloatTensor(y_train_dict[task].reshape(-1, 1)).to(device)
        y_test_tensors[task] = torch.FloatTensor(y_test_dict[task].reshape(-1, 1)).to(device)

    X_train_t = torch.FloatTensor(X_train_np).to(device)
    X_test_t = torch.FloatTensor(X_test_np).to(device)

    # --- 2. Model i Optymalizator ---
    model = MTL_NN_Hybrid(input_dim=X_train_np.shape[1], reg_tasks=reg_tasks, class_tasks=class_tasks).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion_reg = nn.MSELoss(reduction='none') # 'none' pozwala nam ręcznie obsłużyć NaN
    criterion_cls = nn.BCEWithLogitsLoss(reduction='none')

    print("\n--- START TRENINGU (Multi-Task) ---")
    model.train()
    for epoch in range(500):
        optimizer.zero_grad()
        outputs = model(X_train_t)
        total_loss = 0

        # Sumujemy straty ze wszystkich zadań, ignorując NaN
        for task in reg_tasks:
            target = y_train_tensors[task]
            mask = ~torch.isnan(target)
            if mask.any():
                loss = criterion_reg(outputs[task][mask], target[mask]).mean()
                total_loss += loss

        for task in class_tasks:
            target = y_train_tensors[task]
            mask = ~torch.isnan(target)
            if mask.any():
                loss = criterion_cls(outputs[task][mask], target[mask]).mean()
                total_loss += loss

        total_loss.backward()
        optimizer.step()

        if epoch % 20 == 0:
            print(f"  Epoka {epoch:3d} | Total Loss: {total_loss.item():.4f}")

    # --- 3. Ewaluacja ---
    print("\n--- EWALUACJA ---")
    model.eval()
    all_metrics = {"reg_tasks": {}, "class_tasks": {}}

    with torch.no_grad():
        preds_dict = model(X_test_t)

        for task in reg_tasks:
            y_true = y_test_dict[task].flatten()
            mask = ~np.isnan(y_true)
            if not mask.any(): continue

            # Odwracamy skalowanie
            p_scaled = preds_dict[task].cpu().numpy()
            p_unscaled = scalers[task].inverse_transform(p_scaled).flatten()


            mse = mean_squared_error(y_true[mask], p_unscaled[mask])
            mae = mean_absolute_error(y_true[mask], p_unscaled[mask]) # <-- Dodaj tę linię
            r2 = r2_score(y_true[mask], p_unscaled[mask])

            all_metrics["reg_tasks"][task] = {
               "rmse": np.sqrt(mse),
               "mae": mae,  # <-- I tę linię
               "r2": r2
             }

        for task in class_tasks:
            y_true = y_test_dict[task].flatten()
            mask = ~np.isnan(y_true)
            if not mask.any(): continue

            p_probs  = torch.sigmoid(preds_dict[task]).cpu().numpy().flatten()
            p_labels = (p_probs[mask] > 0.5).astype(int)
            try:
                auroc = roc_auc_score(y_true[mask], p_probs[mask])
            except ValueError:
                auroc = 0.5

            all_metrics["class_tasks"][task] = {
                "accuracy": accuracy_score(y_true[mask], p_labels),
                "f1":       f1_score(y_true[mask], p_labels),
                "auroc":    auroc,
            }

    return all_metrics

In [9]:
def train_MTL_hybrid_uniform_average(X_train_np, X_test_np, y_train_dict, y_test_dict, reg_tasks, class_tasks):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    scalers = {}
    y_train_tensors = {}
    y_test_tensors = {}

    for task in reg_tasks:
        scalers[task] = StandardScaler()
        train_vals = y_train_dict[task].reshape(-1, 1)
        mask = ~np.isnan(train_vals).flatten()

        y_train_scaled = np.full_like(train_vals, np.nan)
        if mask.any():
            y_train_scaled[mask] = scalers[task].fit_transform(train_vals[mask])

        y_train_tensors[task] = torch.FloatTensor(y_train_scaled).to(device)
        y_test_tensors[task] = torch.FloatTensor(y_test_dict[task].reshape(-1, 1)).to(device)

    for task in class_tasks:
        y_train_tensors[task] = torch.FloatTensor(y_train_dict[task].reshape(-1, 1)).to(device)
        y_test_tensors[task] = torch.FloatTensor(y_test_dict[task].reshape(-1, 1)).to(device)

    X_train_t = torch.FloatTensor(X_train_np).to(device)
    X_test_t = torch.FloatTensor(X_test_np).to(device)

    model = MTL_NN_Hybrid(input_dim=X_train_np.shape[1], reg_tasks=reg_tasks, class_tasks=class_tasks).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion_reg = nn.MSELoss(reduction='none')
    criterion_cls = nn.BCEWithLogitsLoss(reduction='none')

    print("\n--- START TRENINGU (Multi-Task - Uniform Average) ---")
    model.train()
    for epoch in range(500):
        optimizer.zero_grad()
        outputs = model(X_train_t)

        task_losses = []

        for task in reg_tasks:
            target = y_train_tensors[task]
            mask = ~torch.isnan(target)
            if mask.any():
                loss = criterion_reg(outputs[task][mask], target[mask]).mean()
                task_losses.append(loss)

        for task in class_tasks:
            target = y_train_tensors[task]
            mask = ~torch.isnan(target)
            if mask.any():
                loss = criterion_cls(outputs[task][mask], target[mask]).mean()
                task_losses.append(loss)

        if task_losses:
            total_loss = torch.stack(task_losses).mean() # Uniform average
        else:
            total_loss = torch.tensor(0.0, device=device, requires_grad=True)

        total_loss.backward()
        optimizer.step()

        if epoch % 20 == 0:
            print(f"  Epoka {epoch:3d} | Total Loss: {total_loss.item():.4f}")

    print("\n--- EWALUACJA ---")
    model.eval()
    all_metrics = {"reg_tasks": {}, "class_tasks": {}}

    with torch.no_grad():
        preds_dict = model(X_test_t)

        for task in reg_tasks:
            y_true = y_test_dict[task].flatten()
            mask = ~np.isnan(y_true)
            if not mask.any(): continue

            p_scaled = preds_dict[task].cpu().numpy()
            p_unscaled = scalers[task].inverse_transform(p_scaled).flatten()

            mse = mean_squared_error(y_true[mask], p_unscaled[mask])
            mae = mean_absolute_error(y_true[mask], p_unscaled[mask])
            r2 = r2_score(y_true[mask], p_unscaled[mask])

            all_metrics["reg_tasks"][task] = {
               "rmse": np.sqrt(mse),
               "mae": mae,
               "r2": r2
             }

        for task in class_tasks:
            y_true = y_test_dict[task].flatten()
            mask = ~np.isnan(y_true)
            if not mask.any(): continue

            p_probs  = torch.sigmoid(preds_dict[task]).cpu().numpy().flatten()
            p_labels = (p_probs[mask] > 0.5).astype(int)
            try:
                auroc = roc_auc_score(y_true[mask], p_probs[mask])
            except ValueError:
                auroc = 0.5

            all_metrics["class_tasks"][task] = {
                "accuracy": accuracy_score(y_true[mask], p_labels),
                "f1":       f1_score(y_true[mask], p_labels),
                "auroc":    auroc,
            }

    return all_metrics

In [10]:
def train_MTL_hybrid_uncertainty_weighting(X_train_np, X_test_np, y_train_dict, y_test_dict, reg_tasks, class_tasks):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    scalers = {}
    y_train_tensors = {}
    y_test_tensors = {}

    for task in reg_tasks:
        scalers[task] = StandardScaler()
        train_vals = y_train_dict[task].reshape(-1, 1)
        mask = ~np.isnan(train_vals).flatten()

        y_train_scaled = np.full_like(train_vals, np.nan)
        if mask.any():
            y_train_scaled[mask] = scalers[task].fit_transform(train_vals[mask])

        y_train_tensors[task] = torch.FloatTensor(y_train_scaled).to(device)
        y_test_tensors[task] = torch.FloatTensor(y_test_dict[task].reshape(-1, 1)).to(device)

    for task in class_tasks:
        y_train_tensors[task] = torch.FloatTensor(y_train_dict[task].reshape(-1, 1)).to(device)
        y_test_tensors[task] = torch.FloatTensor(y_test_dict[task].reshape(-1, 1)).to(device)

    X_train_t = torch.FloatTensor(X_train_np).to(device)
    X_test_t = torch.FloatTensor(X_test_np).to(device)

    model = MTL_NN_Hybrid(input_dim=X_train_np.shape[1], reg_tasks=reg_tasks, class_tasks=class_tasks).to(device) #zmienione

    # Learnable parameters for task uncertainty
    log_vars = nn.ParameterDict()
    for task in reg_tasks + class_tasks:
        log_vars[task] = nn.Parameter(torch.zeros(1, device=device)) # Initialize log_sigma_sq to 0

    # Include log_vars in the optimizer
    optimizer = optim.Adam(list(model.parameters()) + list(log_vars.values()), lr=0.001)
    criterion_reg = nn.MSELoss(reduction='none')
    criterion_cls = nn.BCEWithLogitsLoss(reduction='none')

    print("\n--- START TRENINGU (Multi-Task - Uncertainty Weighting) ---")
    model.train()
    for epoch in range(500):
        optimizer.zero_grad()
        outputs = model(X_train_t)
        total_loss = 0

        for task in reg_tasks:
            target = y_train_tensors[task]
            mask = ~torch.isnan(target)
            if mask.any():
                loss = criterion_reg(outputs[task][mask], target[mask])
                precision = torch.exp(-log_vars[task])
                total_loss += torch.mean(0.5 * precision * loss + 0.5 * log_vars[task])

        for task in class_tasks:
            target = y_train_tensors[task]
            mask = ~torch.isnan(target)
            if mask.any():
                loss = criterion_cls(outputs[task][mask], target[mask])
                precision = torch.exp(-log_vars[task])
                total_loss += torch.mean(precision * loss + 0.5 * log_vars[task])

        total_loss.backward()
        optimizer.step()

        if epoch % 20 == 0:
            print(f"  Epoka {epoch:3d} | Total Loss: {total_loss.item():.4f}")
            # print(f"    Log_vars: {{ {', '.join([f'{t}: {log_vars[t].item():.2f}' for t in log_vars])} }}")

    print("\n--- EWALUACJA ---")
    model.eval()
    all_metrics = {"reg_tasks": {}, "class_tasks": {}}

    with torch.no_grad():
        preds_dict = model(X_test_t)

        for task in reg_tasks:
            y_true = y_test_dict[task].flatten()
            mask = ~np.isnan(y_true)
            if not mask.any(): continue

            p_scaled = preds_dict[task].cpu().numpy()
            p_unscaled = scalers[task].inverse_transform(p_scaled).flatten()

            mse = mean_squared_error(y_true[mask], p_unscaled[mask])
            mae = mean_absolute_error(y_true[mask], p_unscaled[mask])
            r2 = r2_score(y_true[mask], p_unscaled[mask])

            all_metrics["reg_tasks"][task] = {
               "rmse": np.sqrt(mse),
               "mae": mae,
               "r2": r2
             }

        for task in class_tasks:
            y_true = y_test_dict[task].flatten()
            mask = ~np.isnan(y_true)
            if not mask.any(): continue

            p_probs  = torch.sigmoid(preds_dict[task]).cpu().numpy().flatten()
            p_labels = (p_probs[mask] > 0.5).astype(int)
            try:
                auroc = roc_auc_score(y_true[mask], p_probs[mask])
            except ValueError:
                auroc = 0.5

            all_metrics["class_tasks"][task] = {
                "accuracy": accuracy_score(y_true[mask], p_labels),
                "f1":       f1_score(y_true[mask], p_labels),
                "auroc":    auroc,
            }

    return all_metrics

In [11]:
import pickle

def load_split_pickle(dataset_name):
    filepath = f"{data_folder}/{dataset_name}_split.pkl"

    with open(filepath, "rb") as f:
        split = pickle.load(f)

    return split["train"], split["test"]

In [12]:
def print_metrics(metrics, task='classification', weight_loss_func_name=None):
    print(f"\n{'='*40}")
    if weight_loss_func_name: # Check if func_name is provided and not None
        print(f"  Loss Weighting: {weight_loss_func_name}")
        print(f"{'='*40}")
    if task == 'classification':
        print(f"  Accuracy : {metrics['test_metrics']['accuracy']:.4f}")
        print(f"  F1       : {metrics['test_metrics']['f1']:.4f}")
        print(f"  AUROC    : {metrics['test_metrics']['auroc']:.4f}")
    else:
        print(f"  RMSE     : {metrics['test_metrics']['rmse']:.4f}")
        print(f"  MAE      : {metrics['test_metrics']['mae']:.4f}")
        print(f"  R²       : {metrics['test_metrics']['r2']:.4f}")
    print(f"{'='*40}\n")


def save_metrics(metrics, dataset_name, filepath, task='classification', weight_loss_func_name=None, endpoint_group_name=None):
    with open(filepath, 'a') as f:
        f.write(f"\n{'='*40}\n")
        f.write(f"Endpoint    : {dataset_name}\n")
        if endpoint_group_name:
            f.write(f"Tasks       : {endpoint_group_name}\n")
        if weight_loss_func_name:
            f.write(f"Loss Weighting: {weight_loss_func_name}\n")
        f.write(f"{'='*40}\n")
        if task == 'classification':
            f.write(f"  Accuracy : {metrics['test_metrics']['accuracy']:.4f}\n")
            f.write(f"  F1       : {metrics['test_metrics']['f1']:.4f}\n")
            f.write(f"  AUROC    : {metrics['test_metrics']['auroc']:.4f}\n")
        else:
            f.write(f"  RMSE     : {metrics['test_metrics']['rmse']:.4f}\n")
            f.write(f"  MAE      : {metrics['test_metrics']['mae']:.4f}\n")
            f.write(f"  R²       : {metrics['test_metrics']['r2']:.4f}\n")
        f.write(f"{'='*40}\n")

In [13]:
import numpy as np
import pandas as pd

def prepare_mtl_data_embeddings(df_list, task_names, dataset_names):
    """
    df_list: lista DataFrame ze splitów (np. [train_caco2, train_lipo])
    task_names: lista nazw zadań w modelu (np. ['Caco2_Wang', 'Lipophilicity_AZ'])
    dataset_names: lista nazw zbiorów odpowiadająca nazwom plików z embeddingami (np. ['Caco2_Wang', 'Lipophilicity_AstraZeneca'])
    """
    # 1. Zebranie wszystkich unikalnych struktur SMILES z przekazanych splitów
    all_drugs = set()
    for df in df_list:
        valid = df['Drug'].dropna().astype(str).unique()
        all_drugs.update(valid)

    # 2. Wczytanie embeddingów ze wszystkich powiązanych plików CSV dla danej konfiguracji
    # Budujemy jeden zbiorczy słownik mapujący Drug (SMILES) -> wektor embeddingu
    drug_to_emb = {}
    for d_name in dataset_names:
        file_path = f"{data_folder}/{d_name}_MoLFormer_embeddings.csv"
        try:
            emb_df = pd.read_csv(file_path)
            feature_cols = [c for c in emb_df.columns if c.startswith('emb_')]
            for _, row in emb_df.iterrows():
                drug_to_emb[str(row['Drug'])] = row[feature_cols].values.astype(np.float32)
        except Exception as e:
            print(f"Błąd podczas ładowania embeddingów dla {d_name}: {e}")

    # 3. Filtrowanie master_list tylko do cząsteczek, które posiadają embeddingi
    safe_master_list = [drug for drug in sorted(list(all_drugs)) if drug in drug_to_emb]
    drug_to_idx = {drug: i for i, drug in enumerate(safe_master_list)}
    n_samples = len(safe_master_list)

    # 4. Tworzenie macierzy X_features z embeddingów
    emb_dim = len(next(iter(drug_to_emb.values()))) if drug_to_emb else 768
    X_features = np.zeros((n_samples, emb_dim), dtype=np.float32)
    for i, drug in enumerate(safe_master_list):
        X_features[i] = drug_to_emb[drug]

    # 5. Mapowanie etykiet y (dokładnie tak samo jak w wersji z deskryptorami)
    y_dict = {}
    for df, task in zip(df_list, task_names):
        y_vec = np.full((n_samples, 1), np.nan, dtype=np.float32)
        mapping = dict(zip(df['Drug'].astype(str), df['Y']))

        for drug, val in mapping.items():
            if drug in drug_to_idx and not pd.isna(val):
                y_vec[drug_to_idx[drug]] = val
        y_dict[task] = y_vec

    return X_features, y_dict

# Test1: Half Life (Obach) + Clearance Hepatocyte (AstraZeneca)

Test pierwszy sprawdzający czy klirens wątrobowy (Clearance Hepatocyte) pomoże w predykcji okresu półtrwania (Half Life) - oba zadania są bezpośrednio powiązane mechanistycznie. Sprawdzamy czy MTL da lepsze wyniki niż STL.

In [14]:
reg_tasks = ['Half_Life_Obach', 'Clearance_Hepatocyte_AZ']
class_tasks = [] # Dodaj tu zadania klasyfikacyjne
filepath = "/content/drive/MyDrive/MLDD - ADMET/mtl_results_eliminacja_embeddings_500epochs.txt"

dataset_names = ['Half_Life_Obach', 'Clearance_Hepatocyte_AZ']

# 1. Ładowanie danych
train_halflife, test_halflife = load_split_pickle('Half_Life_Obach')
train_clear, test_clear = load_split_pickle('Clearance_Hepatocyte_AZ')

# 2. Przygotowanie danych MTL
X_train_mtl, y_train_dict = prepare_mtl_data_embeddings([train_halflife, train_clear], reg_tasks, dataset_names)
X_test_mtl,  y_test_dict  = prepare_mtl_data_embeddings([test_halflife, test_clear],  reg_tasks, dataset_names)


# Define all loss weighting schemes to test
loss_schemes = [
    (train_MTL_hybrid_wl_sum, "Weighted Loss Sum"),
    (train_MTL_hybrid_uniform_average, "Uniform Average"),
    (train_MTL_hybrid_uncertainty_weighting, "Uncertainty Weighting")
]

# Loop through each loss weighting scheme
for train_func, scheme_name in loss_schemes:


    # 3. Uruchomienie treningu
    results = train_func(
        X_train_mtl,
        X_test_mtl,
        y_train_dict,
        y_test_dict,
        reg_tasks,
        class_tasks=class_tasks
    )

    # 4. RAPORTOWANIE I ZAPISYWANIE WYNIKÓW

    print("\n" + "="*40)
    print(f"       PODSUMOWANIE MODELU MTL - {scheme_name}")
    print("="*40)


    endpoint_group = str(reg_tasks + class_tasks)

    # Przetwarzanie zadań regresyjnych
    for task_name, task_metrics in results['reg_tasks'].items():
        # Przygotowanie formatu pod Twoje funkcje (opakowanie w 'test_metrics')
        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Regresja): {task_name}")
        print_metrics(formatted_wrapper, task='regression', weight_loss_func_name=scheme_name)
        save_metrics(formatted_wrapper, task_name, filepath, task='regression', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

    # Przetwarzanie zadań klasyfikacyjnych (jeśli istnieją)
    for task_name, task_metrics in results['class_tasks'].items():

        # if 'f1' not in task_metrics:
        #     task_metrics['f1'] = 0.0  # Wypełnienie, jeśli funkcja wymaga f1

        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Klasyfikacja): {task_name}")
        print_metrics(formatted_wrapper, task='classification', weight_loss_func_name=scheme_name)
        save_metrics( formatted_wrapper, task_name, filepath, task='classification', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

print(f"\nWszystkie wyniki zostały zapisane w: {filepath}")


--- START TRENINGU (Multi-Task) ---
  Epoka   0 | Total Loss: 2.3572
  Epoka  20 | Total Loss: 1.8116
  Epoka  40 | Total Loss: 1.2017
  Epoka  60 | Total Loss: 0.7718
  Epoka  80 | Total Loss: 0.4441
  Epoka 100 | Total Loss: 0.2835
  Epoka 120 | Total Loss: 0.1821
  Epoka 140 | Total Loss: 0.1058
  Epoka 160 | Total Loss: 0.0774
  Epoka 180 | Total Loss: 0.0719
  Epoka 200 | Total Loss: 0.0621
  Epoka 220 | Total Loss: 0.0531
  Epoka 240 | Total Loss: 0.0526
  Epoka 260 | Total Loss: 0.0393
  Epoka 280 | Total Loss: 0.0348
  Epoka 300 | Total Loss: 0.0377
  Epoka 320 | Total Loss: 0.0353
  Epoka 340 | Total Loss: 0.0254
  Epoka 360 | Total Loss: 0.0293
  Epoka 380 | Total Loss: 0.0300
  Epoka 400 | Total Loss: 0.0273
  Epoka 420 | Total Loss: 0.0236
  Epoka 440 | Total Loss: 0.0223
  Epoka 460 | Total Loss: 0.0236
  Epoka 480 | Total Loss: 0.0199

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Weighted Loss Sum

Wyniki dla zadania (Regresja): Half_Life_Obach

  Loss Weighting: 

# Test2: Half Life (Obach) + Clearance Hepatocyte (AstraZeneca) + CYP3A4 Inhibition (Veith)

Dokładamy zadanie klasyfikacyjne CYP3A4 Inhibition. Sprawdzamy czy zadanie powiązane mechanistycznie z eliminacją (CYP3A4 metabolizuje 50% leków) wnosi dodatkową informację. Pierwszy test mieszany (regresja + klasyfikacja).

In [15]:
reg_tasks = ['Half_Life_Obach', 'Clearance_Hepatocyte_AZ']
class_tasks = ['CYP3A4_Veith'] # Dodaj tu zadania klasyfikacyjne
filepath = "/content/drive/MyDrive/MLDD - ADMET/mtl_results_eliminacja_embeddings_500epochs.txt"

dataset_names = ['Half_Life_Obach', 'Clearance_Hepatocyte_AZ', 'CYP3A4_Veith']

# 1. Ładowanie danych
train_halflife, test_halflife = load_split_pickle('Half_Life_Obach')
train_clear, test_clear = load_split_pickle('Clearance_Hepatocyte_AZ')
train_cyp, test_cyp = load_split_pickle('CYP3A4_Veith')

# 2. Przygotowanie danych MTL
X_train_mtl, y_train_dict = prepare_mtl_data_embeddings([train_halflife, train_clear, train_cyp], reg_tasks + class_tasks, dataset_names)
X_test_mtl,  y_test_dict  = prepare_mtl_data_embeddings([test_halflife, test_clear, test_cyp],  reg_tasks + class_tasks, dataset_names)

# Define all loss weighting schemes to test
loss_schemes = [
    (train_MTL_hybrid_wl_sum, "Weighted Loss Sum"),
    (train_MTL_hybrid_uniform_average, "Uniform Average"),
    (train_MTL_hybrid_uncertainty_weighting, "Uncertainty Weighting")
]

# Loop through each loss weighting scheme
for train_func, scheme_name in loss_schemes:


    # 3. Uruchomienie treningu
    results = train_func(
        X_train_mtl,
        X_test_mtl,
        y_train_dict,
        y_test_dict,
        reg_tasks,
        class_tasks=class_tasks
    )

    # 4. RAPORTOWANIE I ZAPISYWANIE WYNIKÓW

    print("\n" + "="*40)
    print(f"       PODSUMOWANIE MODELU MTL - {scheme_name}")
    print("="*40)


    endpoint_group = str(reg_tasks + class_tasks)

    # Przetwarzanie zadań regresyjnych
    for task_name, task_metrics in results['reg_tasks'].items():
        # Przygotowanie formatu pod Twoje funkcje (opakowanie w 'test_metrics')
        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Regresja): {task_name}")
        print_metrics(formatted_wrapper, task='regression', weight_loss_func_name=scheme_name)
        save_metrics(formatted_wrapper, task_name, filepath, task='regression', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

    # Przetwarzanie zadań klasyfikacyjnych (jeśli istnieją)
    for task_name, task_metrics in results['class_tasks'].items():

        # if 'f1' not in task_metrics:
        #     task_metrics['f1'] = 0.0  # Wypełnienie, jeśli funkcja wymaga f1

        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Klasyfikacja): {task_name}")
        print_metrics(formatted_wrapper, task='classification', weight_loss_func_name=scheme_name)
        save_metrics( formatted_wrapper, task_name, filepath, task='classification', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

print(f"\nWszystkie wyniki zostały zapisane w: {filepath}")


--- START TRENINGU (Multi-Task) ---
  Epoka   0 | Total Loss: 3.7707
  Epoka  20 | Total Loss: 2.2440
  Epoka  40 | Total Loss: 1.6643
  Epoka  60 | Total Loss: 1.1930
  Epoka  80 | Total Loss: 0.8488
  Epoka 100 | Total Loss: 0.6075
  Epoka 120 | Total Loss: 0.4811
  Epoka 140 | Total Loss: 0.4402
  Epoka 160 | Total Loss: 0.3546
  Epoka 180 | Total Loss: 0.2968
  Epoka 200 | Total Loss: 0.2556
  Epoka 220 | Total Loss: 0.1803
  Epoka 240 | Total Loss: 0.1407
  Epoka 260 | Total Loss: 0.1134
  Epoka 280 | Total Loss: 0.0943
  Epoka 300 | Total Loss: 0.0785
  Epoka 320 | Total Loss: 0.0771
  Epoka 340 | Total Loss: 0.0712
  Epoka 360 | Total Loss: 0.0598
  Epoka 380 | Total Loss: 0.0643
  Epoka 400 | Total Loss: 0.0513
  Epoka 420 | Total Loss: 0.0415
  Epoka 440 | Total Loss: 0.0390
  Epoka 460 | Total Loss: 0.0383
  Epoka 480 | Total Loss: 0.0406

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Weighted Loss Sum

Wyniki dla zadania (Regresja): Half_Life_Obach

  Loss Weighting: 

# Test3: Half Life (Obach) + Clearance Hepatocyte (AstraZeneca) + CYP3A4 Inhibition (Veith) + VDss (Lombardo)

Dokładamy VDss - kompletny zestaw eliminacyjny (4 endpointy biologicznie powiązane). Sprawdzamy czy pełne pokrycie domeny daje najlepsze wyniki, czy w pewnym momencie wyniki przestają się poprawiać.

In [16]:
reg_tasks = ['Half_Life_Obach', 'Clearance_Hepatocyte_AZ', 'VDss_Lombardo']
class_tasks = ['CYP3A4_Veith'] # Dodaj tu zadania klasyfikacyjne
filepath = "/content/drive/MyDrive/MLDD - ADMET/mtl_results_eliminacja_embeddings_500epochs.txt"

dataset_names = ['Half_Life_Obach', 'Clearance_Hepatocyte_AZ', 'VDss_Lombardo', 'CYP3A4_Veith']

# 1. Ładowanie danych
train_halflife, test_halflife = load_split_pickle('Half_Life_Obach')
train_clear, test_clear = load_split_pickle('Clearance_Hepatocyte_AZ')
train_vdss, test_vdss = load_split_pickle('VDss_Lombardo')
train_cyp, test_cyp = load_split_pickle('CYP3A4_Veith')

# 2. Przygotowanie danych MTL
X_train_mtl, y_train_dict = prepare_mtl_data_embeddings([train_halflife, train_clear, train_vdss, train_cyp], reg_tasks + class_tasks, dataset_names)
X_test_mtl,  y_test_dict  = prepare_mtl_data_embeddings([test_halflife, test_clear, test_vdss, test_cyp],  reg_tasks + class_tasks, dataset_names)

# Define all loss weighting schemes to test
loss_schemes = [
    (train_MTL_hybrid_wl_sum, "Weighted Loss Sum"),
    (train_MTL_hybrid_uniform_average, "Uniform Average"),
    (train_MTL_hybrid_uncertainty_weighting, "Uncertainty Weighting")
]

# Loop through each loss weighting scheme
for train_func, scheme_name in loss_schemes:


    # 3. Uruchomienie treningu
    results = train_func(
        X_train_mtl,
        X_test_mtl,
        y_train_dict,
        y_test_dict,
        reg_tasks,
        class_tasks=class_tasks
    )

    # 4. RAPORTOWANIE I ZAPISYWANIE WYNIKÓW

    print("\n" + "="*40)
    print(f"       PODSUMOWANIE MODELU MTL - {scheme_name}")
    print("="*40)


    endpoint_group = str(reg_tasks + class_tasks)

    # Przetwarzanie zadań regresyjnych
    for task_name, task_metrics in results['reg_tasks'].items():
        # Przygotowanie formatu pod Twoje funkcje (opakowanie w 'test_metrics')
        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Regresja): {task_name}")
        print_metrics(formatted_wrapper, task='regression', weight_loss_func_name=scheme_name)
        save_metrics(formatted_wrapper, task_name, filepath, task='regression', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

    # Przetwarzanie zadań klasyfikacyjnych (jeśli istnieją)
    for task_name, task_metrics in results['class_tasks'].items():

        # if 'f1' not in task_metrics:
        #     task_metrics['f1'] = 0.0  # Wypełnienie, jeśli funkcja wymaga f1

        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Klasyfikacja): {task_name}")
        print_metrics(formatted_wrapper, task='classification', weight_loss_func_name=scheme_name)
        save_metrics( formatted_wrapper, task_name, filepath, task='classification', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

print(f"\nWszystkie wyniki zostały zapisane w: {filepath}")


--- START TRENINGU (Multi-Task) ---
  Epoka   0 | Total Loss: 4.9166
  Epoka  20 | Total Loss: 3.0430
  Epoka  40 | Total Loss: 2.1913
  Epoka  60 | Total Loss: 1.3285
  Epoka  80 | Total Loss: 0.7606
  Epoka 100 | Total Loss: 0.5888
  Epoka 120 | Total Loss: 0.5322
  Epoka 140 | Total Loss: 0.4763
  Epoka 160 | Total Loss: 0.4087
  Epoka 180 | Total Loss: 0.3503
  Epoka 200 | Total Loss: 0.2970
  Epoka 220 | Total Loss: 0.2390
  Epoka 240 | Total Loss: 0.2545
  Epoka 260 | Total Loss: 0.1533
  Epoka 280 | Total Loss: 0.1767
  Epoka 300 | Total Loss: 0.1049
  Epoka 320 | Total Loss: 0.0946
  Epoka 340 | Total Loss: 0.0935
  Epoka 360 | Total Loss: 0.0795
  Epoka 380 | Total Loss: 0.0855
  Epoka 400 | Total Loss: 0.0776
  Epoka 420 | Total Loss: 0.0662
  Epoka 440 | Total Loss: 0.0671
  Epoka 460 | Total Loss: 0.0906
  Epoka 480 | Total Loss: 0.0650

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Weighted Loss Sum

Wyniki dla zadania (Regresja): Half_Life_Obach

  Loss Weighting: 

# Test4: Clearance Hepatocyte (AstraZeneca) + CYP3A4 Inhibition (Veith)

Sprawdzamy parę regresja + klasyfikacja - oba zadania bezpośrednio związane z metabolizmem wątrobowym. Jednocześnie porównujemy STL vs MTL.

In [17]:
reg_tasks = ['Clearance_Hepatocyte_AZ']
class_tasks = ['CYP3A4_Veith'] # Dodaj tu zadania klasyfikacyjne
filepath = "/content/drive/MyDrive/MLDD - ADMET/mtl_results_eliminacja_embeddings_500epochs.txt"

dataset_names = ['Clearance_Hepatocyte_AZ', 'CYP3A4_Veith']

# 1. Ładowanie danych
train_clear, test_clear = load_split_pickle('Clearance_Hepatocyte_AZ')
train_cyp, test_cyp = load_split_pickle('CYP3A4_Veith')

# 2. Przygotowanie danych MTL
X_train_mtl, y_train_dict = prepare_mtl_data_embeddings([train_clear, train_cyp], reg_tasks + class_tasks, dataset_names)
X_test_mtl,  y_test_dict  = prepare_mtl_data_embeddings([test_clear, test_cyp],  reg_tasks + class_tasks, dataset_names)

# Define all loss weighting schemes to test
loss_schemes = [
    (train_MTL_hybrid_wl_sum, "Weighted Loss Sum"),
    (train_MTL_hybrid_uniform_average, "Uniform Average"),
    (train_MTL_hybrid_uncertainty_weighting, "Uncertainty Weighting")
]

# Loop through each loss weighting scheme
for train_func, scheme_name in loss_schemes:


    # 3. Uruchomienie treningu
    results = train_func(
        X_train_mtl,
        X_test_mtl,
        y_train_dict,
        y_test_dict,
        reg_tasks,
        class_tasks=class_tasks
    )

    # 4. RAPORTOWANIE I ZAPISYWANIE WYNIKÓW

    print("\n" + "="*40)
    print(f"       PODSUMOWANIE MODELU MTL - {scheme_name}")
    print("="*40)

    endpoint_group = str(reg_tasks + class_tasks)

    # Przetwarzanie zadań regresyjnych
    for task_name, task_metrics in results['reg_tasks'].items():
        # Przygotowanie formatu pod Twoje funkcje (opakowanie w 'test_metrics')
        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Regresja): {task_name}")
        print_metrics(formatted_wrapper, task='regression', weight_loss_func_name=scheme_name)
        save_metrics(formatted_wrapper, task_name, filepath, task='regression', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

    # Przetwarzanie zadań klasyfikacyjnych (jeśli istnieją)
    for task_name, task_metrics in results['class_tasks'].items():

        # if 'f1' not in task_metrics:
        #     task_metrics['f1'] = 0.0  # Wypełnienie, jeśli funkcja wymaga f1

        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Klasyfikacja): {task_name}")
        print_metrics(formatted_wrapper, task='classification', weight_loss_func_name=scheme_name)
        save_metrics( formatted_wrapper, task_name, filepath, task='classification', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

print(f"\nWszystkie wyniki zostały zapisane w: {filepath}")


--- START TRENINGU (Multi-Task) ---
  Epoka   0 | Total Loss: 2.1146
  Epoka  20 | Total Loss: 1.5609
  Epoka  40 | Total Loss: 1.2904
  Epoka  60 | Total Loss: 1.1248
  Epoka  80 | Total Loss: 0.9516
  Epoka 100 | Total Loss: 0.7650
  Epoka 120 | Total Loss: 0.6012
  Epoka 140 | Total Loss: 0.5024
  Epoka 160 | Total Loss: 0.4281
  Epoka 180 | Total Loss: 0.3572
  Epoka 200 | Total Loss: 0.3019
  Epoka 220 | Total Loss: 0.2379
  Epoka 240 | Total Loss: 0.1803
  Epoka 260 | Total Loss: 0.1357
  Epoka 280 | Total Loss: 0.1127
  Epoka 300 | Total Loss: 0.0812
  Epoka 320 | Total Loss: 0.0652
  Epoka 340 | Total Loss: 0.0553
  Epoka 360 | Total Loss: 0.0450
  Epoka 380 | Total Loss: 0.0387
  Epoka 400 | Total Loss: 0.0388
  Epoka 420 | Total Loss: 0.0341
  Epoka 440 | Total Loss: 0.0275
  Epoka 460 | Total Loss: 0.0275
  Epoka 480 | Total Loss: 0.0243

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Weighted Loss Sum

Wyniki dla zadania (Regresja): Clearance_Hepatocyte_AZ

  Loss Wei

# Test5: Half Life (Obach) + CYP3A4 Inhibition (Veith)

Sprawdzamy czy CYP3A4 (klasyfikacja) pomoże regresji Half Life - są to zadania powiązane pośrednio przez metabolizm enzymu CYP3A4.

In [18]:
reg_tasks = ['Half_Life_Obach']
class_tasks = ['CYP3A4_Veith'] # Dodaj tu zadania klasyfikacyjne
filepath = "/content/drive/MyDrive/MLDD - ADMET/mtl_results_eliminacja_embeddings_500epochs.txt"

dataset_names = ['Half_Life_Obach', 'CYP3A4_Veith']

# 1. Ładowanie danych
train_halflife, test_halflife = load_split_pickle('Half_Life_Obach')
train_cyp, test_cyp = load_split_pickle('CYP3A4_Veith')

# 2. Przygotowanie danych MTL
X_train_mtl, y_train_dict = prepare_mtl_data_embeddings([train_halflife, train_cyp], reg_tasks + class_tasks, dataset_names)
X_test_mtl,  y_test_dict  = prepare_mtl_data_embeddings([test_halflife, test_cyp],  reg_tasks + class_tasks, dataset_names)

# Define all loss weighting schemes to test
loss_schemes = [
    (train_MTL_hybrid_wl_sum, "Weighted Loss Sum"),
    (train_MTL_hybrid_uniform_average, "Uniform Average"),
    (train_MTL_hybrid_uncertainty_weighting, "Uncertainty Weighting")
]

# Loop through each loss weighting scheme
for train_func, scheme_name in loss_schemes:


    # 3. Uruchomienie treningu
    results = train_func(
        X_train_mtl,
        X_test_mtl,
        y_train_dict,
        y_test_dict,
        reg_tasks,
        class_tasks=class_tasks
    )

    # 4. RAPORTOWANIE I ZAPISYWANIE WYNIKÓW

    print("\n" + "="*40)
    print(f"       PODSUMOWANIE MODELU MTL - {scheme_name}")
    print("="*40)


    endpoint_group = str(reg_tasks + class_tasks)

    # Przetwarzanie zadań regresyjnych
    for task_name, task_metrics in results['reg_tasks'].items():
        # Przygotowanie formatu pod Twoje funkcje (opakowanie w 'test_metrics')
        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Regresja): {task_name}")
        print_metrics(formatted_wrapper, task='regression', weight_loss_func_name=scheme_name)
        save_metrics(formatted_wrapper, task_name, filepath, task='regression', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

    # Przetwarzanie zadań klasyfikacyjnych (jeśli istnieją)
    for task_name, task_metrics in results['class_tasks'].items():

        # if 'f1' not in task_metrics:
        #     task_metrics['f1'] = 0.0  # Wypełnienie, jeśli funkcja wymaga f1

        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Klasyfikacja): {task_name}")
        print_metrics(formatted_wrapper, task='classification', weight_loss_func_name=scheme_name)
        save_metrics( formatted_wrapper, task_name, filepath, task='classification', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

print(f"\nWszystkie wyniki zostały zapisane w: {filepath}")


--- START TRENINGU (Multi-Task) ---
  Epoka   0 | Total Loss: 2.0248
  Epoka  20 | Total Loss: 1.3867
  Epoka  40 | Total Loss: 0.9789
  Epoka  60 | Total Loss: 0.7773
  Epoka  80 | Total Loss: 0.5913
  Epoka 100 | Total Loss: 0.4751
  Epoka 120 | Total Loss: 0.4439
  Epoka 140 | Total Loss: 0.4068
  Epoka 160 | Total Loss: 0.3562
  Epoka 180 | Total Loss: 0.3227
  Epoka 200 | Total Loss: 0.2742
  Epoka 220 | Total Loss: 0.2047
  Epoka 240 | Total Loss: 0.1920
  Epoka 260 | Total Loss: 0.1379
  Epoka 280 | Total Loss: 0.1031
  Epoka 300 | Total Loss: 0.0769
  Epoka 320 | Total Loss: 0.0628
  Epoka 340 | Total Loss: 0.0525
  Epoka 360 | Total Loss: 0.0457
  Epoka 380 | Total Loss: 0.0406
  Epoka 400 | Total Loss: 0.0274
  Epoka 420 | Total Loss: 0.0256
  Epoka 440 | Total Loss: 0.0208
  Epoka 460 | Total Loss: 0.0251
  Epoka 480 | Total Loss: 0.0231

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Weighted Loss Sum

Wyniki dla zadania (Regresja): Half_Life_Obach

  Loss Weighting: 

# Test6: Half Life (Obach) + AMES Mutagenicity

Sprawdzamy łączenie z niepowiązanym biologicznie zadaniem (AMES - mutagenność). Oczekujemy transferu negatywnego.

In [19]:
reg_tasks = ['Half_Life_Obach']
class_tasks = ['AMES'] # Dodaj tu zadania klasyfikacyjne
filepath = "/content/drive/MyDrive/MLDD - ADMET/mtl_results_eliminacja_embeddings_500epochs.txt"

dataset_names = ['Half_Life_Obach', 'AMES']

# 1. Ładowanie danych
train_halflife, test_halflife = load_split_pickle('Half_Life_Obach')
train_ames, test_ames = load_split_pickle('AMES')

# 2. Przygotowanie danych MTL
X_train_mtl, y_train_dict = prepare_mtl_data_embeddings([train_halflife, train_ames], reg_tasks + class_tasks, dataset_names)
X_test_mtl,  y_test_dict  = prepare_mtl_data_embeddings([test_halflife, test_ames],  reg_tasks + class_tasks, dataset_names)

# Define all loss weighting schemes to test
loss_schemes = [
    (train_MTL_hybrid_wl_sum, "Weighted Loss Sum"),
    (train_MTL_hybrid_uniform_average, "Uniform Average"),
    (train_MTL_hybrid_uncertainty_weighting, "Uncertainty Weighting")
]

# Loop through each loss weighting scheme
for train_func, scheme_name in loss_schemes:


    # 3. Uruchomienie treningu
    results = train_func(
        X_train_mtl,
        X_test_mtl,
        y_train_dict,
        y_test_dict,
        reg_tasks,
        class_tasks=class_tasks
    )

    # 4. RAPORTOWANIE I ZAPISYWANIE WYNIKÓW

    print("\n" + "="*40)
    print(f"       PODSUMOWANIE MODELU MTL - {scheme_name}")
    print("="*40)


    endpoint_group = str(reg_tasks + class_tasks)

    # Przetwarzanie zadań regresyjnych
    for task_name, task_metrics in results['reg_tasks'].items():
        # Przygotowanie formatu pod Twoje funkcje (opakowanie w 'test_metrics')
        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Regresja): {task_name}")
        print_metrics(formatted_wrapper, task='regression', weight_loss_func_name=scheme_name)
        save_metrics(formatted_wrapper, task_name, filepath, task='regression', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

    # Przetwarzanie zadań klasyfikacyjnych (jeśli istnieją)
    for task_name, task_metrics in results['class_tasks'].items():

        # if 'f1' not in task_metrics:
        #     task_metrics['f1'] = 0.0  # Wypełnienie, jeśli funkcja wymaga f1

        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Klasyfikacja): {task_name}")
        print_metrics(formatted_wrapper, task='classification', weight_loss_func_name=scheme_name)
        save_metrics( formatted_wrapper, task_name, filepath, task='classification', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

print(f"\nWszystkie wyniki zostały zapisane w: {filepath}")


--- START TRENINGU (Multi-Task) ---
  Epoka   0 | Total Loss: 1.8951
  Epoka  20 | Total Loss: 1.5698
  Epoka  40 | Total Loss: 1.0673
  Epoka  60 | Total Loss: 0.8512
  Epoka  80 | Total Loss: 0.6996
  Epoka 100 | Total Loss: 0.5585
  Epoka 120 | Total Loss: 0.4402
  Epoka 140 | Total Loss: 0.3933
  Epoka 160 | Total Loss: 0.3178
  Epoka 180 | Total Loss: 0.2289
  Epoka 200 | Total Loss: 0.1780
  Epoka 220 | Total Loss: 0.1306
  Epoka 240 | Total Loss: 0.0839
  Epoka 260 | Total Loss: 0.0632
  Epoka 280 | Total Loss: 0.0650
  Epoka 300 | Total Loss: 0.0427
  Epoka 320 | Total Loss: 0.0385
  Epoka 340 | Total Loss: 0.0402
  Epoka 360 | Total Loss: 0.0204
  Epoka 380 | Total Loss: 0.0340
  Epoka 400 | Total Loss: 0.0181
  Epoka 420 | Total Loss: 0.0178
  Epoka 440 | Total Loss: 0.0199
  Epoka 460 | Total Loss: 0.0313
  Epoka 480 | Total Loss: 0.0121

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Weighted Loss Sum

Wyniki dla zadania (Regresja): Half_Life_Obach

  Loss Weighting: 

# Test7: Half Life (Obach) + Clearance Hepatocyte (AstraZeneca) + CYP3A4 Inhibition (Veith) + VDss (Lombardo) + AMES Mutagenicity

Pełny zestaw 5 endpointów: 4 powiązane biologicznie + 1 niepowiązany (AMES). Sprawdzamy łączny efekt - czy AMES zaszkodzi pozostałym, czy pełne pokrycie wygrywa.

In [20]:
reg_tasks = ['Half_Life_Obach', 'Clearance_Hepatocyte_AZ', 'VDss_Lombardo']
class_tasks = ['CYP3A4_Veith', 'AMES'] # Dodaj tu zadania klasyfikacyjne
filepath = "/content/drive/MyDrive/MLDD - ADMET/mtl_results_eliminacja_embeddings_500epochs.txt"

dataset_names = ['Half_Life_Obach', 'Clearance_Hepatocyte_AZ', 'VDss_Lombardo', 'CYP3A4_Veith', 'AMES']

# 1. Ładowanie danych
train_halflife, test_halflife = load_split_pickle('Half_Life_Obach')
train_clear, test_clear = load_split_pickle('Clearance_Hepatocyte_AZ')
train_vdss, test_vdss = load_split_pickle('VDss_Lombardo')
train_cyp, test_cyp = load_split_pickle('CYP3A4_Veith')
train_ames, test_ames = load_split_pickle('AMES')

# 2. Przygotowanie danych MTL
X_train_mtl, y_train_dict = prepare_mtl_data_embeddings([train_halflife, train_clear, train_vdss, train_cyp, train_ames], reg_tasks + class_tasks, dataset_names)
X_test_mtl,  y_test_dict  = prepare_mtl_data_embeddings([test_halflife, test_clear, test_vdss, test_cyp, test_ames],  reg_tasks + class_tasks, dataset_names)

# Define all loss weighting schemes to test
loss_schemes = [
    (train_MTL_hybrid_wl_sum, "Weighted Loss Sum"),
    (train_MTL_hybrid_uniform_average, "Uniform Average"),
    (train_MTL_hybrid_uncertainty_weighting, "Uncertainty Weighting")
]

# Loop through each loss weighting scheme
for train_func, scheme_name in loss_schemes:


    # 3. Uruchomienie treningu
    results = train_func(
        X_train_mtl,
        X_test_mtl,
        y_train_dict,
        y_test_dict,
        reg_tasks,
        class_tasks=class_tasks
    )

    # 4. RAPORTOWANIE I ZAPISYWANIE WYNIKÓW

    print("\n" + "="*40)
    print(f"       PODSUMOWANIE MODELU MTL - {scheme_name}")
    print("="*40)


    endpoint_group = str(reg_tasks + class_tasks)

    # Przetwarzanie zadań regresyjnych
    for task_name, task_metrics in results['reg_tasks'].items():
        # Przygotowanie formatu pod Twoje funkcje (opakowanie w 'test_metrics')
        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Regresja): {task_name}")
        print_metrics(formatted_wrapper, task='regression', weight_loss_func_name=scheme_name)
        save_metrics(formatted_wrapper, task_name, filepath, task='regression', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

    # Przetwarzanie zadań klasyfikacyjnych (jeśli istnieją)
    for task_name, task_metrics in results['class_tasks'].items():

        # if 'f1' not in task_metrics:
        #     task_metrics['f1'] = 0.0  # Wypełnienie, jeśli funkcja wymaga f1

        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Klasyfikacja): {task_name}")
        print_metrics(formatted_wrapper, task='classification', weight_loss_func_name=scheme_name)
        save_metrics( formatted_wrapper, task_name, filepath, task='classification', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

print(f"\nWszystkie wyniki zostały zapisane w: {filepath}")


--- START TRENINGU (Multi-Task) ---
  Epoka   0 | Total Loss: 4.9146
  Epoka  20 | Total Loss: 3.5169
  Epoka  40 | Total Loss: 2.5599
  Epoka  60 | Total Loss: 1.6683
  Epoka  80 | Total Loss: 1.1592
  Epoka 100 | Total Loss: 1.0066
  Epoka 120 | Total Loss: 0.9063
  Epoka 140 | Total Loss: 0.7874
  Epoka 160 | Total Loss: 0.7302
  Epoka 180 | Total Loss: 0.5726
  Epoka 200 | Total Loss: 0.4756
  Epoka 220 | Total Loss: 0.3700
  Epoka 240 | Total Loss: 0.3082
  Epoka 260 | Total Loss: 0.2462
  Epoka 280 | Total Loss: 0.2064
  Epoka 300 | Total Loss: 0.1698
  Epoka 320 | Total Loss: 0.1429
  Epoka 340 | Total Loss: 0.1209
  Epoka 360 | Total Loss: 0.1189
  Epoka 380 | Total Loss: 0.1052
  Epoka 400 | Total Loss: 0.0873
  Epoka 420 | Total Loss: 0.0786
  Epoka 440 | Total Loss: 0.0718
  Epoka 460 | Total Loss: 0.0697
  Epoka 480 | Total Loss: 0.0705

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Weighted Loss Sum

Wyniki dla zadania (Regresja): Half_Life_Obach

  Loss Weighting: 